# Модуль 1 · Разбор исследовательского кейса
## Почему изменение двух параметров дало неожиданный результат?

**Тип:** пошаговое исследование с готовым кодом и интерпретацией.  
**Время:** ориентировочно 40–60 минут.  
**Предпосылки:** базовые понятия процента, среднего значения и графика; программирование не требуется.

Вы подключились к исследованию прочности экспериментального полимерного покрытия. В лаборатории изучают два управляемых условия: температуру термообработки и массовую долю добавки. Четыре испытанных сочетания дали неожиданные значения прочности. Нужно решить, достаточно ли просто увеличить температуру, или влияние температуры зависит от состава.

**Важно:** все числа ниже специально сконструированы для учебного примера. Это НЕ опубликованные экспериментальные измерения и НЕ достоверные прогнозы реального покрытия. Задача учит анализу факторного эксперимента, а не обещает рецептуру материала.

**Результат:** вы сможете объяснить взаимодействие факторов, построить простую модель, проверить арифметику и обосновать, какого эксперимента не хватает для научного вывода.

### 1. Сначала гипотеза, затем код

Инженер предлагает: «Если повышение температуры улучшает прочность, её нужно повысить для всех составов».

Перед расчётом ответьте: может ли изменение температуры давать **разный эффект** при различной доле добавки? Какая пара измерений позволила бы это обнаружить?

Мы проверим не только разницу между двумя средними, но и **изменение самого эффекта**.

Научный вопрос: зависит ли влияние температуры от доли добавки в этих четырёх учебных точках?

In [ ]:
import sys
import math
import json
from pathlib import Path
import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0])
print("Matplotlib:", plt.matplotlib.__version__ if hasattr(plt, "matplotlib") else __import__("matplotlib").__version__)

### 2. Паспорт наблюдений

Фактор A — температура термообработки: 50 °C (код −1), 70 °C (код +1). Фактор B — доля добавки: 1% массы (−1), 3% массы (+1). Отклик — условная прочность на растяжение, МПа.

**Проверка до расчёта:** все четыре комбинации представлены ровно один раз. Отсутствуют повторы; это ограничивает любые заявления об ошибке измерения, значимости и достоверности прогноза.

In [ ]:
runs = [
    {"id": "R1", "A": -1, "B": -1, "temp_C": 50, "additive_mass_pct": 1, "strength_MPa": 43.0},
    {"id": "R2", "A": +1, "B": -1, "temp_C": 70, "additive_mass_pct": 1, "strength_MPa": 47.0},
    {"id": "R3", "A": -1, "B": +1, "temp_C": 50, "additive_mass_pct": 3, "strength_MPa": 37.0},
    {"id": "R4", "A": +1, "B": +1, "temp_C": 70, "additive_mass_pct": 3, "strength_MPa": 73.0},
]
assert len(runs) == 4
assert {(r["A"], r["B"]) for r in runs} == {(-1,-1),(1,-1),(-1,1),(1,1)}
assert all(math.isfinite(r["strength_MPa"]) for r in runs)
for r in runs:
    print(f'{r["id"]}: T={r["temp_C"]} °C, добавка={r["additive_mass_pct"]}% → {r["strength_MPa"]:.1f} МПа')

### 3. Скрытая деталь — температура действует не одинаково

Сравните R1 с R2: доля добавки неизменна, различается только температура. Затем сравните R3 с R4.

Если оба изменения прочности близки, эффект температуры можно считать примерно одинаковым в этих условиях. Если они сильно различаются, появляется **взаимодействие факторов**: изменение одного фактора зависит от уровня другого.

Здесь не требуется сначала знать сложную статистическую теорию. Начинаем с двух честных сравнений при одном неизменном условии.

In [ ]:
by_level = {(r["A"],r["B"]): r["strength_MPa"] for r in runs}
low_additive_temp_effect = by_level[(1,-1)] - by_level[(-1,-1)]
high_additive_temp_effect = by_level[(1,1)] - by_level[(-1,1)]
difference_of_temp_effects = high_additive_temp_effect - low_additive_temp_effect
print("Эффект температуры при 1% добавки:", low_additive_temp_effect, "МПа")
print("Эффект температуры при 3% добавки:", high_additive_temp_effect, "МПа")
print("Разность этих эффектов:", difference_of_temp_effects, "МПа")
assert (low_additive_temp_effect,high_additive_temp_effect,difference_of_temp_effects)==(4.0,36.0,32.0)

**Интерпретация.** В учебных данных повышение температуры даёт +4 МПа при 1% добавки и +36 МПа при 3% добавки. Разность +32 МПа — это сравнение двух условных эффектов, а не погрешность прибора и не оценка статистической значимости.

Попробуйте объяснить результат словами без формулы. Почему совет «просто увеличьте температуру» неполон?

### 4. Четыре точки и факторные эффекты

В двухуровневом полном плане 2² определяют средний эффект A как разность средних отклика при A=+1 и A=−1. Аналогично для B.

Эффект взаимодействия AB в распространённой кодировке — разность средних откликов двух групп, где произведение A·B равно +1 и −1. Он равен половине разности двух условных эффектов A. Важно не путать эти **две разные численные конвенции**.

In [ ]:
def average(values):
    return sum(values) / len(values)

grand_mean = average([r["strength_MPa"] for r in runs])
effect_A = average([r["strength_MPa"] for r in runs if r["A"]==1]) - average([r["strength_MPa"] for r in runs if r["A"]==-1])
effect_B = average([r["strength_MPa"] for r in runs if r["B"]==1]) - average([r["strength_MPa"] for r in runs if r["B"]==-1])
effect_AB = average([r["strength_MPa"] for r in runs if r["A"]*r["B"]==1]) - average([r["strength_MPa"] for r in runs if r["A"]*r["B"]==-1])
print(f"Общий средний отклик: {grand_mean:g} МПа")
print(f"Эффект A: {effect_A:g} МПа, B: {effect_B:g} МПа, AB: {effect_AB:g} МПа")
assert (grand_mean,effect_A,effect_B,effect_AB)==(50.0,20.0,10.0,16.0)
assert abs(effect_AB - difference_of_temp_effects/2) < 1e-12

### 5. Визуализация как проверка мысли

Построим **график взаимодействия**. Если линии соответствующих составов существенно непараллельны, изменения температурных эффектов на двух уровнях добавки неодинаковы.

Это визуальный способ увидеть наблюдаемую разницу, **не** статистическая проверка значимости взаимодействия (повторов пока нет).

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
for b, label in [(-1,"Добавка 1% масс."),(1,"Добавка 3% масс.")]:
    ys=[by_level[(-1,b)],by_level[(1,b)]]
    ax.plot([50,70],ys,marker="o",linewidth=2,label=label)
ax.set(xlabel="Температура термообработки, °C",ylabel="Условная прочность, МПа",title="Влияние температуры зависит от доли добавки (учебные данные)")
ax.set_xticks([50,70])
ax.grid(alpha=.25)
ax.legend()
plt.show()

### 6. Маленькая математическая модель — и большая ловушка

Для кодированных факторов A и B можно написать насыщенную линейную модель:

ŷ = β₀ + βA·A + βB·B + βAB·A·B.

Здесь β₀ — общий средний отклик, а коэффициенты при A, B и AB равны половинам соответствующих эффектов (при данной кодировке). Модель с четырьмя коэффициентами **точно воспроизводит четыре наблюдения**, потому что на четыре точки приходится четыре параметра.

Точное совпадение ещё не доказывает физическую правильность модели. В частности, нет независимой оценки случайной ошибки; за пределами четырёх условий делать предсказания рискованно.

In [ ]:
beta0, betaA, betaB, betaAB = grand_mean, effect_A/2, effect_B/2, effect_AB/2
def predict_coded(a,b):
    return beta0 + betaA*a + betaB*b + betaAB*a*b

for r in runs:
    pred=predict_coded(r["A"],r["B"])
    print(f'{r["id"]}: наблюдение {r["strength_MPa"]:.1f}, модель {pred:.1f} МПа')
    assert math.isclose(pred,r["strength_MPa"],rel_tol=0,abs_tol=1e-12)

n, n_parameters = len(runs), 4
print("Степени свободы остатка:", n - n_parameters)
assert n - n_parameters == 0

**Неожиданный вывод.** Модель безошибочно объяснила все наблюдения и одновременно **не имеет остаточных степеней свободы**. Это пример того, почему совпадение расчёта с учебной таблицей не служит независимой научной валидацией. Нельзя здесь вычислять обычную оценку остаточной дисперсии и объявлять p-value эффектов.

**Следующий эксперимент.** Повторить измерения при каждом из четырёх сочетаний, рандомизировать порядок запуска, зафиксировать время/партию образцов, дополнительно рассмотреть центральную точку. Если цели и ресурсы позволяют, проверить другие уровни и устойчивость эффекта. Не делайте вывод о рецептуре материала по синтетической таблице.

In [ ]:
result={
    "scenario":"Synthetic 2^2 materials experiment; not measured data",
    "units":{"temperature":"degC","additive":"mass_pct","strength":"MPa"},
    "observations":runs,
    "effects_MPa":{"A":effect_A,"B":effect_B,"AB_factorial":effect_AB},
    "difference_of_conditional_A_effects_MPa":difference_of_temp_effects,
    "model_coefficients":{"b0":beta0,"bA":betaA,"bB":betaB,"bAB":betaAB},
    "n_observations":n,"residual_degrees_of_freedom":n-n_parameters,
    "limitations":["synthetic_data","one_observation_per_cell","no_error_estimate","no_external_validation"],
}
output_dir=Path("outputs")
output_dir.mkdir(exist_ok=True)
output_file=output_dir/"module01_guided_case_results.json"
output_file.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding="utf-8")
print("Паспорт анализа:",output_file)

### 7. Обновление вашей карты знаний

Добавьте и **обоснуйте** отношения:
- фактор эксперимента → задаёт → условие получения наблюдения;
- взаимодействие факторов → означает → зависимость эффекта A от уровня B (в заданном диапазоне);
- насыщенная модель → может точно воспроизводить → обучающую таблицу, но **не даёт** независимой оценки ошибки без повторов;
- гипотеза → проверяется → новым экспериментом.

**Контрольные вопросы.** (1) Почему 32 МПа и 16 МПа не противоречат друг другу? (2) Почему отсутствие повторов мешает оценить значимость? (3) Что убедило бы вас в устойчивости эффекта при новых измерениях?

**Следующий блокнот:** вы получите новые данные из другого кейса, столкнётесь с разными единицами и сами составите обоснованный план повторного эксперимента.